# 01 · Universe exploration

What the price panel actually contains, before any factor touches it.

Universe *construction* (the market-cap / ADV / listing filters) is Matt's
A5-A6. This notebook only checks that what arrives is usable.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
pd.set_option('display.width', 140)

# Synthetic market today. When Matt's A3 cache exists, this becomes:
#     from quant.factors.base import load_panel
#     panel = load_panel()
from tests.conftest import make_synthetic_panel
panel, truth = make_synthetic_panel()
panel


## Coverage

Missing data is not neutral — a name absent on a date silently
drops out of that date's cross-section and changes what everything else
is ranked against.


In [ ]:
coverage = panel.adj_close.notna().mean(axis=1)
print(f'mean ticker coverage per date: {coverage.mean():.1%}')
print(f'worst date: {coverage.min():.1%} on {coverage.idxmin().date()}')
print(f'tickers never priced: {(panel.adj_close.notna().sum() == 0).sum()}')


## Liquidity

The ADV distribution is what the section 10 universe filter cuts on,
and what caps position size later.


In [ ]:
adv = (panel.adj_close * panel.volume).iloc[-20:].mean()
print(adv.describe().apply(lambda x: f'${x:,.0f}'))
print()
for threshold in (1e6, 5e6, 2e7):
    print(f'  names above ${threshold:,.0f} ADV: {(adv > threshold).sum()} / {len(adv)}')


## Return sanity

Anything past about +/-50% in a day is a split that was not adjusted,
not a real move. One of those poisons a whole cross-section.


In [ ]:
rets = panel.returns()
print(f'daily return sd (median across names): {rets.std().median():.3%}')
extreme = (rets.abs() > 0.5).sum().sum()
print(f'|return| > 50% observations: {extreme}')
print(f'non-positive prices: {(panel.adj_close <= 0).sum().sum()}')
